First Install PySpark

In [1]:
%pip install pyspark, pandas, numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: C:\Users\chatu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: 'pyspark,': Expected end or semicolon (after name and no valid version specifier)
    pyspark,
           ^


Now we can check the version of the PySpark and it'll also check that PySpark is installed correctly or not.

In [2]:
import pyspark

print(pyspark.__version__)

4.2.0


Now here we are creating a session, here we are using (local[*]) it means means Spark will use the available CPU cores on our computer.

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BankingCustomerAnalytics")
    .master("local[*]")
    .getOrCreate()
)

print(spark.version)

4.2.0


Here we are load the data

In [4]:
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/customers.csv")
)

customers.show(5)
customers.printSchema()

+-----------+---+------+---------+------------+---------------------+-------------+-----------+
|customer_id|age|gender|     city|account_type|customer_tenure_years|annual_income|signup_date|
+-----------+---+------+---------+------------+---------------------+-------------+-----------+
| CUST000001| 56|Female|    Delhi|     Savings|                  7.1|       180000| 2011-07-28|
| CUST000002| 69|  Male|Ahmedabad|     Current|                 12.5|       180000| 2023-09-20|
| CUST000003| 46|  Male|  Kolkata|     Savings|                 12.7|       180000| 2015-03-28|
| CUST000004| 32|Female|   Mumbai|     Savings|                  5.6|       180000| 2016-10-02|
| CUST000005| 60|  Male|   Jaipur|     Savings|                  8.4|       180000| 2016-06-09|
+-----------+---+------+---------+------------+---------------------+-------------+-----------+
only showing top 5 rows
root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullabl

Here in the output we can see all the columns from customers data with top 5 rows, also we can see the data type of every column

Now here I'm going to see total number of columns and rows in the data.

In [5]:
print("Customers:", customers.count())
print("Columns:", len(customers.columns))

Customers: 50000
Columns: 8


Total rows are 50000 and total columns are 8.

Here I'm loading the transaction data.

In [6]:
transactions = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/transactions.csv")
)

transactions.show(5)
transactions.printSchema()

print("Transactions:", transactions.count())

+--------------+-----------+----------------+----------------+------------------+-----------------+--------------+------------------+
|transaction_id|customer_id|transaction_date|transaction_type|transaction_amount|merchant_category|       channel|transaction_status|
+--------------+-----------+----------------+----------------+------------------+-----------------+--------------+------------------+
|   TXN00000001| CUST048549|      2025-08-20|           Debit|           1052.26|           Travel|           UPI|           Success|
|   TXN00000002| CUST003074|      2025-05-27|           Debit|           1796.29|       Healthcare|          Card|           Success|
|   TXN00000003| CUST001456|      2025-04-18|          Credit|            200.52|            Other|           UPI|           Success|
|   TXN00000004| CUST049119|      2025-07-12|           Debit|           1257.07|        Utilities|Mobile Banking|           Success|
|   TXN00000005| CUST046864|      2025-04-25|           Debit|

In this data we can see all columns, top 5 rows and data type of every column.

|| From here our main analysis of the data begin ||

here we can see some statistical values for every column, it'll help us to understand our data.

In [7]:
customers.describe().show()

+-------+-----------+------------------+------+---------+------------+---------------------+-----------------+
|summary|customer_id|               age|gender|     city|account_type|customer_tenure_years|    annual_income|
+-------+-----------+------------------+------+---------+------------+---------------------+-----------------+
|  count|      50000|             50000| 50000|    50000|       50000|                50000|            50000|
|   mean|       NULL|          46.51676|  NULL|     NULL|        NULL|    7.751459999999987|     180960.30558|
| stddev|       NULL|16.709248429599104|  NULL|     NULL|        NULL|    4.184770353849594|9369.045673923096|
|    min| CUST000001|                18|Female|Ahmedabad|     Current|                  0.5|           180000|
|    max| CUST050000|                75| Other|     Pune|     Savings|                 15.0|           479358|
+-------+-----------+------------------+------+---------+------------+---------------------+-----------------+



In [8]:
transactions.describe().show()

+-------+--------------+-----------+----------------+------------------+-----------------+-------+------------------+
|summary|transaction_id|customer_id|transaction_type|transaction_amount|merchant_category|channel|transaction_status|
+-------+--------------+-----------+----------------+------------------+-----------------+-------+------------------+
|  count|       1000500|    1000500|         1000500|           1000500|           999500|1000500|           1000500|
|   mean|          NULL|       NULL|            NULL| 6546.342206156927|             NULL|   NULL|              NULL|
| stddev|          NULL|       NULL|            NULL| 68598.19718238841|             NULL|   NULL|              NULL|
|    min|   TXN00000001| CUST000001|          Credit|              50.0|           Dining|    ATM|            Failed|
|    max|   TXN01000000| CUST050000|           Debit|         1499957.1|        Utilities|    UPI|           Success|
+-------+--------------+-----------+----------------+---

Here I'm looking for some selected columns from the transactions data, top 20 rows. What kinf of valur it contains.

In [9]:
transactions.select(
    "transaction_type",
    "merchant_category",
    "channel",
    "transaction_status"
).show(20)

+----------------+-----------------+----------------+------------------+
|transaction_type|merchant_category|         channel|transaction_status|
+----------------+-----------------+----------------+------------------+
|           Debit|           Travel|             UPI|           Success|
|           Debit|       Healthcare|            Card|           Success|
|          Credit|            Other|             UPI|           Success|
|           Debit|        Utilities|  Mobile Banking|           Success|
|           Debit|        Utilities|             ATM|           Success|
|          Credit|            Other|          Branch|           Success|
|          Credit|        Utilities|             UPI|           Success|
|           Debit|           Dining|             UPI|           Success|
|           Debit|           Dining|  Mobile Banking|           Success|
|          Credit|    Entertainment|Internet Banking|           Success|
|           Debit|        Utilities|             UP

Now I'm looking for null values.

In [10]:
from pyspark.sql.functions import col, sum

transactions.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in transactions.columns
]).show()

+--------------+-----------+----------------+----------------+------------------+-----------------+-------+------------------+
|transaction_id|customer_id|transaction_date|transaction_type|transaction_amount|merchant_category|channel|transaction_status|
+--------------+-----------+----------------+----------------+------------------+-----------------+-------+------------------+
|             0|          0|               0|               0|                 0|             1000|      0|                 0|
+--------------+-----------+----------------+----------------+------------------+-----------------+-------+------------------+



Now here we can see there are total 1000 missing values present in merchant_category column.

So now we know that we have 1000 missing values in merchant_category, let's replace them with (unknown)

In [11]:
transactions_clean = transactions.fillna(
    {"merchant_category": "Unknown"}
)

Now let's check again if any missing value present in the data.

In [12]:
transactions_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in transactions_clean.columns
]).show()

+--------------+-----------+----------------+----------------+------------------+-----------------+-------+------------------+
|transaction_id|customer_id|transaction_date|transaction_type|transaction_amount|merchant_category|channel|transaction_status|
+--------------+-----------+----------------+----------------+------------------+-----------------+-------+------------------+
|             0|          0|               0|               0|                 0|                0|      0|                 0|
+--------------+-----------+----------------+----------------+------------------+-----------------+-------+------------------+



Here we can see now there is no missing value present in any column.

Here I'm checking duplicate transaction ids

In [13]:
duplicate_count = (
    transactions
    .groupBy("transaction_id")
    .count()
    .filter("count > 1")
    .count()
)

print("Duplicate transaction IDs:", duplicate_count)

Duplicate transaction IDs: 500


There are total 500 duplicate transaction ids.

Now we can see top 10 id's which are duplicate

In [14]:
transactions.groupBy("transaction_id") \
    .count() \
    .filter("count > 1") \
    .show(10)

+--------------+-----+
|transaction_id|count|
+--------------+-----+
|   TXN00051565|    2|
|   TXN00026707|    2|
|   TXN00043275|    2|
|   TXN00006299|    2|
|   TXN00051334|    2|
|   TXN00055400|    2|
|   TXN00040811|    2|
|   TXN00021220|    2|
|   TXN00064291|    2|
|   TXN00030321|    2|
+--------------+-----+
only showing top 10 rows


So now we know that we have 1000 missing values iRemove these duplicate teansaction ids

In [15]:
transactions_clean = transactions_clean.dropDuplicates(
    ["transaction_id"]
)

Let's check how many rows present before and after this.

In [16]:
print("Original rows:", transactions.count())
print("Clean rows:", transactions_clean.count())

Original rows: 1000500
Clean rows: 1000000


Now here we can see previously there was total 1000500 rows but now we remove those 500 duplicate rows and nrw count is 1000000

In [17]:
transactions_clean.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- merchant_category: string (nullable = false)
 |-- channel: string (nullable = true)
 |-- transaction_status: string (nullable = true)



In [18]:
transactions_clean.show(5)

+--------------+-----------+----------------+----------------+------------------+-----------------+-------+------------------+
|transaction_id|customer_id|transaction_date|transaction_type|transaction_amount|merchant_category|channel|transaction_status|
+--------------+-----------+----------------+----------------+------------------+-----------------+-------+------------------+
|   TXN00000001| CUST048549|      2025-08-20|           Debit|           1052.26|           Travel|    UPI|           Success|
|   TXN00000003| CUST001456|      2025-04-18|          Credit|            200.52|            Other|    UPI|           Success|
|   TXN00000054| CUST037367|      2025-06-24|          Credit|           3144.44|        Education|    UPI|           Success|
|   TXN00000056| CUST026108|      2025-10-05|          Credit|           1036.02|            Other|   Card|           Success|
|   TXN00000061| CUST017786|      2025-06-11|           Debit|            692.27|    Entertainment|    ATM|    

In [19]:
print("Final transaction count:", transactions_clean.count())

Final transaction count: 1000000


Now let's check for customers data

In [20]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- account_type: string (nullable = true)
 |-- customer_tenure_years: double (nullable = true)
 |-- annual_income: integer (nullable = true)
 |-- signup_date: date (nullable = true)



In [21]:
customers.show(5)

+-----------+---+------+---------+------------+---------------------+-------------+-----------+
|customer_id|age|gender|     city|account_type|customer_tenure_years|annual_income|signup_date|
+-----------+---+------+---------+------------+---------------------+-------------+-----------+
| CUST000001| 56|Female|    Delhi|     Savings|                  7.1|       180000| 2011-07-28|
| CUST000002| 69|  Male|Ahmedabad|     Current|                 12.5|       180000| 2023-09-20|
| CUST000003| 46|  Male|  Kolkata|     Savings|                 12.7|       180000| 2015-03-28|
| CUST000004| 32|Female|   Mumbai|     Savings|                  5.6|       180000| 2016-10-02|
| CUST000005| 60|  Male|   Jaipur|     Savings|                  8.4|       180000| 2016-06-09|
+-----------+---+------+---------+------------+---------------------+-------------+-----------+
only showing top 5 rows


Now let's joint both dataset

In [22]:
customer_transactions = transactions_clean.join(
    customers,
    on="customer_id",
    how="inner"
)

I'm using here inner joint

In [23]:
customer_transactions.show(5)

+-----------+--------------+----------------+----------------+------------------+-----------------+-------+------------------+---+------+---------+------------+---------------------+-------------+-----------+
|customer_id|transaction_id|transaction_date|transaction_type|transaction_amount|merchant_category|channel|transaction_status|age|gender|     city|account_type|customer_tenure_years|annual_income|signup_date|
+-----------+--------------+----------------+----------------+------------------+-----------------+-------+------------------+---+------+---------+------------+---------------------+-------------+-----------+
| CUST048549|   TXN00000001|      2025-08-20|           Debit|           1052.26|           Travel|    UPI|           Success| 65|  Male|    Delhi|      Salary|                  5.4|       180000| 2011-05-20|
| CUST001456|   TXN00000003|      2025-04-18|          Credit|            200.52|            Other|    UPI|           Success| 68|Female|Bengaluru|     Current|    

In [24]:
print("Joined rows:", customer_transactions.count())

Joined rows: 1000000


Here I'm importing some metrics

In [25]:
from pyspark.sql.functions import (
    col, sum, count, avg, min, max,
    round, month, year
)

In [26]:
customer_summary = (
    customer_transactions
    .groupBy("customer_id")
    .agg(
        count("transaction_id").alias("transaction_count"),
        round(sum("transaction_amount"), 2).alias("total_transaction_value"),
        round(avg("transaction_amount"), 2).alias("avg_transaction_value"),
        round(min("transaction_amount"), 2).alias("min_transaction_value"),
        round(max("transaction_amount"), 2).alias("max_transaction_value")
    )
)

In [27]:
customer_summary.show(10)

+-----------+-----------------+-----------------------+---------------------+---------------------+---------------------+
|customer_id|transaction_count|total_transaction_value|avg_transaction_value|min_transaction_value|max_transaction_value|
+-----------+-----------------+-----------------------+---------------------+---------------------+---------------------+
| CUST000129|               21|               54943.78|              2616.37|               379.08|             13221.42|
| CUST003436|               27|               54787.29|              2029.16|               173.54|              7846.02|
| CUST040063|               21|               30357.23|              1445.58|               118.45|              4348.77|
| CUST002253|               23|               22934.36|               997.15|                 50.0|              3945.84|
| CUST020810|               22|               47451.56|              2156.89|               291.03|              7627.46|
| CUST033592|           

Now looking for top customers

In [28]:
top_customers = (
    customer_summary
    .orderBy(col("total_transaction_value").desc())
)

top_customers.show(10)

+-----------+-----------------+-----------------------+---------------------+---------------------+---------------------+
|customer_id|transaction_count|total_transaction_value|avg_transaction_value|min_transaction_value|max_transaction_value|
+-----------+-----------------+-----------------------+---------------------+---------------------+---------------------+
| CUST007843|               34|             2995891.34|             88114.45|               133.85|           1471258.17|
| CUST040263|               27|             2945417.51|            109089.54|               287.85|           1252531.46|
| CUST036326|               27|             2880171.85|            106673.03|                55.33|           1449541.55|
| CUST013069|               32|             2875975.18|             89874.22|                82.44|            1424035.0|
| CUST002043|               27|             2870698.01|            106322.15|               128.82|           1459431.92|
| CUST014512|           

Now looking for which banking channels are most frequently used

In [29]:
channel_analysis = (
    transactions_clean
    .groupBy("channel")
    .agg(
        count("transaction_id").alias("transactions"),
        round(sum("transaction_amount"), 2).alias("transaction_value"),
        round(avg("transaction_amount"), 2).alias("avg_transaction")
    )
    .orderBy(col("transaction_value").desc())
)

channel_analysis.show()

+----------------+------------+-----------------+---------------+
|         channel|transactions|transaction_value|avg_transaction|
+----------------+------------+-----------------+---------------+
|             UPI|      320005|  2.12032702816E9|        6625.92|
|  Mobile Banking|      199509|  1.29034716997E9|        6467.61|
|            Card|      149640|   9.8835106779E8|        6604.86|
|Internet Banking|      150433|   9.6850565028E8|        6438.12|
|             ATM|      120395|   7.6669365908E8|        6368.15|
|          Branch|       60018|     4.13925042E8|        6896.68|
+----------------+------------+-----------------+---------------+



Here we can see UPI is the most used banking channel

Now I'm looking for month wise transactions

In [30]:
monthly_transactions = (
    transactions_clean
    .groupBy(
        year("transaction_date").alias("year"),
        month("transaction_date").alias("month")
    )
    .agg(
        count("transaction_id").alias("transaction_count"),
        round(sum("transaction_amount"), 2).alias("transaction_value")
    )
    .orderBy("year", "month")
)

monthly_transactions.show(20)

+----+-----+-----------------+-----------------+
|year|month|transaction_count|transaction_value|
+----+-----+-----------------+-----------------+
|2025|    1|            84864|   5.4749086871E8|
|2025|    2|            76765|   4.8058399353E8|
|2025|    3|            85054|   5.6015405725E8|
|2025|    4|            81735|   5.3850598505E8|
|2025|    5|            84715|   5.3503601111E8|
|2025|    6|            81701|   5.6624371355E8|
|2025|    7|            85148|   5.6869732247E8|
|2025|    8|            85041|   5.6552966048E8|
|2025|    9|            82347|   5.4953205778E8|
|2025|   10|            85484|   5.4111188851E8|
|2025|   11|            81884|   5.4466327669E8|
|2025|   12|            85262|   5.5060078215E8|
+----+-----+-----------------+-----------------+



Now I'm doing customer segmentation

In [31]:
from pyspark.sql.functions import when

In [32]:
customer_segments = (
    customer_summary
    .withColumn(
        "customer_segment",
        when(col("transaction_count") >= 30, "High Activity")
        .when(col("transaction_count") >= 15, "Medium Activity")
        .otherwise("Low Activity")
    )
)

In [33]:
customer_segments.groupBy("customer_segment").count().show()

+----------------+-----+
|customer_segment|count|
+----------------+-----+
|    Low Activity| 5277|
| Medium Activity|43614|
|   High Activity| 1109|
+----------------+-----+



Here we can see total number of customer for all three segments.

Here I'm looking fro each customer's avg transaction, Standard deviation and total number of tranactions

In [34]:
from pyspark.sql.functions import avg, stddev, round, col

customer_stats = (
    transactions_clean
    .groupBy("customer_id")
    .agg(
        round(avg("transaction_amount"), 2).alias("avg_transaction"),
        round(stddev("transaction_amount"), 2).alias("std_transaction"),
        count("transaction_id").alias("transaction_count")
    )
)

customer_stats.show(10)

+-----------+---------------+---------------+-----------------+
|customer_id|avg_transaction|std_transaction|transaction_count|
+-----------+---------------+---------------+-----------------+
| CUST000129|        2616.37|        3295.64|               21|
| CUST003436|        2029.16|        2275.26|               27|
| CUST040063|        1445.58|        1472.04|               21|
| CUST002253|         997.15|         912.66|               23|
| CUST020810|        2156.89|        1997.47|               22|
| CUST033592|        4555.66|        6997.75|               20|
| CUST028041|        1770.01|        1779.76|               22|
| CUST018139|        1745.61|        1582.48|               22|
| CUST016018|         5506.2|       16734.39|               25|
| CUST009667|        2096.08|        1822.65|               25|
+-----------+---------------+---------------+-----------------+
only showing top 10 rows


In [35]:
transactions_with_stats = transactions_clean.join(
    customer_stats,
    on="customer_id",
    how="left"
)

transactions_with_stats.show(5)

+-----------+--------------+----------------+----------------+------------------+-----------------+-------+------------------+---------------+---------------+-----------------+
|customer_id|transaction_id|transaction_date|transaction_type|transaction_amount|merchant_category|channel|transaction_status|avg_transaction|std_transaction|transaction_count|
+-----------+--------------+----------------+----------------+------------------+-----------------+-------+------------------+---------------+---------------+-----------------+
| CUST015996|   TXN00000021|      2025-02-21|           Debit|           3748.25|         Shopping|    UPI|           Success|         2603.7|        4063.63|               22|
| CUST036443|   TXN00000110|      2025-08-30|           Debit|           1414.96|             Fuel|    ATM|           Success|       42811.96|      176907.85|               19|
| CUST037112|   TXN00000019|      2025-09-02|           Debit|            466.59|        Utilities|    ATM|        

In [36]:
from pyspark.sql.functions import avg, stddev, round, col

customer_stats = (
    transactions_clean
    .groupBy("customer_id")
    .agg(
        round(avg("transaction_amount"), 2).alias("avg_transaction"),
        round(stddev("transaction_amount"), 2).alias("std_transaction"),
        count("transaction_id").alias("transaction_count")
    )
)

customer_stats.show(10)

+-----------+---------------+---------------+-----------------+
|customer_id|avg_transaction|std_transaction|transaction_count|
+-----------+---------------+---------------+-----------------+
| CUST000129|        2616.37|        3295.64|               21|
| CUST003436|        2029.16|        2275.26|               27|
| CUST040063|        1445.58|        1472.04|               21|
| CUST002253|         997.15|         912.66|               23|
| CUST020810|        2156.89|        1997.47|               22|
| CUST033592|        4555.66|        6997.75|               20|
| CUST028041|        1770.01|        1779.76|               22|
| CUST018139|        1745.61|        1582.48|               22|
| CUST016018|         5506.2|       16734.39|               25|
| CUST009667|        2096.08|        1822.65|               25|
+-----------+---------------+---------------+-----------------+
only showing top 10 rows


I'm looking for Z-Score here

In [37]:
transactions_with_zscore = transactions_with_stats.withColumn(
    "z_score",
    (col("transaction_amount") - col("avg_transaction")) /
    col("std_transaction")
)

In [38]:
transactions_with_zscore.select(
    "transaction_id",
    "customer_id",
    "transaction_amount",
    "avg_transaction",
    "std_transaction",
    "z_score"
).show(10)

+--------------+-----------+------------------+---------------+---------------+--------------------+
|transaction_id|customer_id|transaction_amount|avg_transaction|std_transaction|             z_score|
+--------------+-----------+------------------+---------------+---------------+--------------------+
|   TXN00000194| CUST009896|           9154.99|        2220.52|        2610.77|    2.65610145665838|
|   TXN00000021| CUST015996|           3748.25|         2603.7|        4063.63| 0.28165704062623814|
|   TXN00000061| CUST017786|            692.27|        2041.06|        2093.97| -0.6441305271804276|
|   TXN00000037| CUST019401|            377.31|        1725.85|        1758.33| -0.7669436340163678|
|   TXN00000110| CUST036443|           1414.96|       42811.96|      176907.85|-0.23400318301307713|
|   TXN00000019| CUST037112|            466.59|        2002.18|        1547.46| -0.9923293655409511|
|   TXN00000213| CUST037334|            994.59|        1578.09|        2054.77|-0.283973388

Here we can see Z-score for every customer with transaction amount, avg transaction amount and std transation amount.

I use here Z-score>3 for anomaly flag.

In [39]:
transactions_with_anomaly = transactions_with_zscore.withColumn(
    "anomaly_flag",
    when(col("z_score") > 3, "Anomaly")
    .otherwise("Normal")
)

In [40]:
transactions_with_anomaly.groupBy(
    "anomaly_flag"
).count().show()

+------------+------+
|anomaly_flag| count|
+------------+------+
|     Anomaly| 26975|
|      Normal|973025|
+------------+------+



Here we can see total Anomaly and normal customers

Now looking for highest-risk transactions

In [41]:
anomalies = (
    transactions_with_anomaly
    .filter(col("anomaly_flag") == "Anomaly")
    .orderBy(col("z_score").desc())
)

anomalies.select(
    "transaction_id",
    "customer_id",
    "transaction_date",
    "transaction_amount",
    "avg_transaction",
    "z_score",
    "channel",
    "merchant_category"
).show(20)

+--------------+-----------+----------------+------------------+---------------+------------------+--------------+-----------------+
|transaction_id|customer_id|transaction_date|transaction_amount|avg_transaction|           z_score|       channel|merchant_category|
+--------------+-----------+----------------+------------------+---------------+------------------+--------------+-----------------+
|   TXN00359499| CUST043536|      2025-11-02|         341134.87|       10196.08| 6.163427909475835|           UPI|    Entertainment|
|   TXN00390058| CUST014631|      2025-02-20|        1284127.72|       35959.84| 6.001633494150348|           UPI|        Utilities|
|   TXN00536992| CUST045533|      2025-07-15|         812671.22|       24150.76| 5.917719639452492|           ATM|             Fuel|
|   TXN00221444| CUST025618|      2025-12-22|        1368994.72|       39141.81| 5.917698536716127|           UPI|           Travel|
|   TXN00107863| CUST026593|      2025-04-08|         470112.53|     

Now here we can see customers which have unusual spending, channels which have more anomalies, merchant categories which are involved.

Now Analyze anomalies by channel.

In [42]:
anomaly_channel = (
    transactions_with_anomaly
    .filter(col("anomaly_flag") == "Anomaly")
    .groupBy("channel")
    .agg(
        count("transaction_id").alias("anomaly_count"),
        round(sum("transaction_amount"), 2).alias("anomaly_value")
    )
    .orderBy(col("anomaly_count").desc())
)

anomaly_channel.show()

+----------------+-------------+---------------+
|         channel|anomaly_count|  anomaly_value|
+----------------+-------------+---------------+
|             UPI|         8591|1.52772713089E9|
|  Mobile Banking|         5430| 9.1359801884E8|
|Internet Banking|         4062|  6.775084283E8|
|            Card|         4040| 7.1321973952E8|
|             ATM|         3248| 5.4387881262E8|
|          Branch|         1604| 2.9776410851E8|
+----------------+-------------+---------------+



We can see Mobile Banking and UPI represented the highest number of unusual transactions.

Analyze anomalies by merchant category

In [43]:
anomaly_category = (
    transactions_with_anomaly
    .filter(col("anomaly_flag") == "Anomaly")
    .groupBy("merchant_category")
    .agg(
        count("transaction_id").alias("anomaly_count"),
        round(sum("transaction_amount"), 2).alias("anomaly_value")
    )
    .orderBy(col("anomaly_count").desc())
)

anomaly_category.show()

+-----------------+-------------+--------------+
|merchant_category|anomaly_count| anomaly_value|
+-----------------+-------------+--------------+
|    Entertainment|         2783|4.5566520502E8|
|        Education|         2742|4.7448096781E8|
|         Shopping|         2721|4.5748981475E8|
|             Fuel|         2720|5.0160036192E8|
|           Travel|         2683| 4.495916163E8|
|           Dining|         2683|4.6929525279E8|
|        Groceries|         2681|4.7342089311E8|
|       Healthcare|         2659| 4.665778662E8|
|            Other|         2645|4.4910452649E8|
|        Utilities|         2628|4.7160534349E8|
|          Unknown|           30|     4864390.8|
+-----------------+-------------+--------------+



Here we can see all the merchant category, anomaly count and anomaly value.

In [44]:
total_value = transactions_clean.select(
    round(sum("transaction_amount"), 2)
).collect()[0][0]

anomaly_value = anomalies.select(
    round(sum("transaction_amount"), 2)
).collect()[0][0]

anomaly_percentage = (anomaly_value / total_value) * 100

print("Total transaction value:", total_value)
print("Anomalous transaction value:", anomaly_value)
print("Anomaly value percentage:", anomaly_percentage, "%")

Total transaction value: 6548149617.28
Anomalous transaction value: 4673696238.68
Anomaly value percentage: 71.37430437365879 %


In [45]:
transactions_with_anomaly.groupBy("anomaly_flag").agg(
    count("transaction_id").alias("transaction_count"),
    round(sum("transaction_amount"), 2).alias("transaction_value")
).show()

+------------+-----------------+-----------------+
|anomaly_flag|transaction_count|transaction_value|
+------------+-----------------+-----------------+
|     Anomaly|            26975|  4.67369623868E9|
|      Normal|           973025|   1.8744533786E9|
+------------+-----------------+-----------------+



In [46]:
transactions_with_zscore.select(
    "z_score"
).describe().show()

+-------+--------------------+
|summary|             z_score|
+-------+--------------------+
|  count|             1000000|
|   mean|-2.73812914547388...|
| stddev|   0.974679919861306|
|    min| -2.0040185044534975|
|    max|   6.163427909475835|
+-------+--------------------+



Calculate Q1 and Q3 by customer

In [47]:
from pyspark.sql.functions import percentile_approx

customer_iqr = (
    transactions_clean
    .groupBy("customer_id")
    .agg(
        percentile_approx(
            "transaction_amount", 0.25
        ).alias("Q1"),

        percentile_approx(
            "transaction_amount", 0.75
        ).alias("Q3")
    )
)

customer_iqr.show(10)

+-----------+------+-------+
|customer_id|    Q1|     Q3|
+-----------+------+-------+
| CUST000007|434.27|3055.06|
| CUST000015|455.06|1268.68|
| CUST000075|592.43|1754.57|
| CUST000082| 576.6|2791.13|
| CUST000084| 187.4|1999.14|
| CUST000099|343.26| 1482.8|
| CUST000111|626.63|1965.06|
| CUST000129|654.33|2527.02|
| CUST000169|370.99|2036.37|
| CUST000182|223.02|1992.28|
+-----------+------+-------+
only showing top 10 rows


Calculate IQR

In [48]:
customer_iqr = customer_iqr.withColumn(
    "IQR",
    col("Q3") - col("Q1")
)

In [49]:
customer_iqr = customer_iqr.withColumn(
    "upper_limit",
    col("Q3") + (1.5 * col("IQR"))
)

In [50]:
transactions_iqr = transactions_clean.join(
    customer_iqr,
    on="customer_id",
    how="left"
)

In [51]:
transactions_iqr = transactions_iqr.withColumn(
    "anomaly_flag",
    when(
        col("transaction_amount") > col("upper_limit"),
        "Anomaly"
    ).otherwise("Normal")
)

In [52]:
transactions_iqr.groupBy("anomaly_flag").agg(
    count("transaction_id").alias("transaction_count"),
    round(sum("transaction_amount"), 2).alias("transaction_value")
).show()

+------------+-----------------+-----------------+
|anomaly_flag|transaction_count|transaction_value|
+------------+-----------------+-----------------+
|     Anomaly|            85726|  5.26663851801E9|
|      Normal|           914274|  1.28151109927E9|
+------------+-----------------+-----------------+



|| Spark SQL Business Analysis ||

Create a temporary SQL view

In [53]:
customer_transactions.createOrReplaceTempView("customer_transactions")

Monthly transaction performance

In [54]:
monthly_analysis = spark.sql("""
SELECT
    YEAR(transaction_date) AS year,
    MONTH(transaction_date) AS month,
    COUNT(transaction_id) AS transaction_count,
    ROUND(SUM(transaction_amount), 2) AS total_transaction_value,
    ROUND(AVG(transaction_amount), 2) AS avg_transaction_value
FROM customer_transactions
GROUP BY
    YEAR(transaction_date),
    MONTH(transaction_date)
ORDER BY
    year,
    month
""")

monthly_analysis.show(20)

+----+-----+-----------------+-----------------------+---------------------+
|year|month|transaction_count|total_transaction_value|avg_transaction_value|
+----+-----+-----------------+-----------------------+---------------------+
|2025|    1|            84864|         5.4749086871E8|              6451.39|
|2025|    2|            76765|         4.8058399353E8|              6260.46|
|2025|    3|            85054|         5.6015405725E8|              6585.86|
|2025|    4|            81735|         5.3850598505E8|              6588.44|
|2025|    5|            84715|         5.3503601111E8|              6315.72|
|2025|    6|            81701|         5.6624371355E8|              6930.68|
|2025|    7|            85148|         5.6869732247E8|              6678.93|
|2025|    8|            85041|         5.6552966048E8|              6650.08|
|2025|    9|            82347|         5.4953205778E8|              6673.37|
|2025|   10|            85484|         5.4111188851E8|              6329.98|

Analyze customer behavior

In [55]:
customer_analysis = spark.sql("""
SELECT
    customer_id,
    COUNT(transaction_id) AS transaction_count,
    ROUND(SUM(transaction_amount), 2) AS total_transaction_value,
    ROUND(AVG(transaction_amount), 2) AS avg_transaction_value
FROM customer_transactions
GROUP BY customer_id
ORDER BY total_transaction_value DESC
""")

customer_analysis.show(20)

+-----------+-----------------+-----------------------+---------------------+
|customer_id|transaction_count|total_transaction_value|avg_transaction_value|
+-----------+-----------------+-----------------------+---------------------+
| CUST007843|               34|             2995891.34|             88114.45|
| CUST040263|               27|             2945417.51|            109089.54|
| CUST036326|               27|             2880171.85|            106673.03|
| CUST013069|               32|             2875975.18|             89874.22|
| CUST002043|               27|             2870698.01|            106322.15|
| CUST014512|               23|             2856839.27|             124210.4|
| CUST010239|               21|              2853488.9|            135880.42|
| CUST003909|               20|             2786739.98|             139337.0|
| CUST013119|               20|             2748562.42|            137428.12|
| CUST003142|               29|             2740833.12|         

Analyze banking channels

In [56]:
channel_analysis = spark.sql("""
SELECT
    channel,
    COUNT(transaction_id) AS transaction_count,
    ROUND(SUM(transaction_amount), 2) AS total_transaction_value,
    ROUND(AVG(transaction_amount), 2) AS avg_transaction_value
FROM customer_transactions
GROUP BY channel
ORDER BY total_transaction_value DESC
""")

channel_analysis.show()

+----------------+-----------------+-----------------------+---------------------+
|         channel|transaction_count|total_transaction_value|avg_transaction_value|
+----------------+-----------------+-----------------------+---------------------+
|             UPI|           320005|        2.12032702816E9|              6625.92|
|  Mobile Banking|           199509|        1.29034716997E9|              6467.61|
|            Card|           149640|         9.8835106779E8|              6604.86|
|Internet Banking|           150433|         9.6850565028E8|              6438.12|
|             ATM|           120395|         7.6669365908E8|              6368.15|
|          Branch|            60018|           4.13925042E8|              6896.68|
+----------------+-----------------+-----------------------+---------------------+



Merchant category analysis

In [57]:
merchant_analysis = spark.sql("""
SELECT
    merchant_category,
    COUNT(transaction_id) AS transaction_count,
    ROUND(SUM(transaction_amount), 2) AS total_transaction_value,
    ROUND(AVG(transaction_amount), 2) AS avg_transaction_value
FROM customer_transactions
GROUP BY merchant_category
ORDER BY total_transaction_value DESC
""")

merchant_analysis.show()

+-----------------+-----------------+-----------------------+---------------------+
|merchant_category|transaction_count|total_transaction_value|avg_transaction_value|
+-----------------+-----------------+-----------------------+---------------------+
|             Fuel|           100233|         6.9071005395E8|              6891.04|
|        Utilities|            99768|         6.6451482581E8|               6660.6|
|        Groceries|            99182|         6.6031999388E8|              6657.66|
|           Dining|            99980|         6.5666070062E8|              6567.92|
|        Education|            99644|         6.5479588041E8|              6571.35|
|       Healthcare|            99980|         6.5195228812E8|              6520.83|
|    Entertainment|           100215|         6.4608997321E8|              6447.04|
|         Shopping|           100657|         6.4343824424E8|              6392.38|
|           Travel|            99776|         6.3688835057E8|              6

Customer segmentation analysis

In [58]:
customer_segments.createOrReplaceTempView("customer_segments")

In [59]:
segment_analysis = spark.sql("""
SELECT
    customer_segment,
    COUNT(customer_id) AS customer_count,
    ROUND(AVG(transaction_count), 2) AS avg_transactions,
    ROUND(AVG(total_transaction_value), 2) AS avg_customer_value
FROM customer_segments
GROUP BY customer_segment
ORDER BY avg_customer_value DESC
""")

segment_analysis.show()

+----------------+--------------+----------------+------------------+
|customer_segment|customer_count|avg_transactions|avg_customer_value|
+----------------+--------------+----------------+------------------+
|   High Activity|          1109|           31.54|         207878.43|
| Medium Activity|         43614|            20.6|         134843.23|
|    Low Activity|          5277|            12.6|          82728.79|
+----------------+--------------+----------------+------------------+



Create Power BI-ready datasets

In [60]:
import os

os.makedirs("../output", exist_ok=True)

Save monthly analysis

In [63]:
%pip install "pandas>=2.2.0"

  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.0 MB 3.4 MB/s eta 0:00:03
   --- ------------------------------------ 0.8/10.0 MB 3.7 MB/s eta 0:00:03
   ---- ----------------------------------- 1.0/10.0 MB 1.5 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/10.0 MB 1.5 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/10.0 MB 1.5 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/10.0 MB 1.5 MB/s eta 0:00:07
   ----- ---------------------------------- 1.3/10.0 MB 838.9 kB/s eta 0:00:11
   ----- ---------------------------------- 1.3/10.0 MB 838.9 kB/s eta 0:00:11
   ----- ---------------------------------- 1.3/10.0 MB 838.9 kB/s eta 0:00:11
   ----- ---------------------------------- 1.3/10.0 MB 838.9 kB/s eta 0:00:11
   ----- ---------------------------------- 1.3/10.0 MB 838.9 kB/s eta 0:00:11
   --


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: C:\Users\chatu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [64]:
import pandas as pd

print(pd.__version__)

3.0.5


In [65]:
import os

os.makedirs("../output", exist_ok=True)

monthly_analysis.toPandas().to_csv(
    "../output/monthly_analysis.csv",
    index=False
)

channel_analysis.toPandas().to_csv(
    "../output/channel_analysis.csv",
    index=False
)

merchant_analysis.toPandas().to_csv(
    "../output/merchant_analysis.csv",
    index=False
)

segment_analysis.toPandas().to_csv(
    "../output/segment_analysis.csv",
    index=False
)

customer_analysis.toPandas().to_csv(
    "../output/customer_analysis.csv",
    index=False
)

print("All files exported successfully!")

C:\Users\chatu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\chatu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
C:\Users\chatu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache

All files exported successfully!


Executive KPI table

In [66]:
kpi_summary = spark.sql("""
SELECT
    COUNT(DISTINCT customer_id) AS total_customers,
    COUNT(transaction_id) AS total_transactions,
    ROUND(SUM(transaction_amount), 2) AS total_transaction_value,
    ROUND(AVG(transaction_amount), 2) AS avg_transaction_value
FROM customer_transactions
""")

kpi_summary.show()

+---------------+------------------+-----------------------+---------------------+
|total_customers|total_transactions|total_transaction_value|avg_transaction_value|
+---------------+------------------+-----------------------+---------------------+
|          50000|           1000000|        6.54814961728E9|              6548.15|
+---------------+------------------+-----------------------+---------------------+



Customer segment table

In [67]:
customer_segments.createOrReplaceTempView("customer_segments")

segment_analysis = spark.sql("""
SELECT
    customer_segment,
    COUNT(customer_id) AS customer_count,
    ROUND(AVG(transaction_count), 2) AS avg_transactions,
    ROUND(AVG(total_transaction_value), 2) AS avg_customer_value
FROM customer_segments
GROUP BY customer_segment
ORDER BY avg_customer_value DESC
""")

segment_analysis.show()

+----------------+--------------+----------------+------------------+
|customer_segment|customer_count|avg_transactions|avg_customer_value|
+----------------+--------------+----------------+------------------+
|   High Activity|          1109|           31.54|         207878.43|
| Medium Activity|         43614|            20.6|         134843.23|
|    Low Activity|          5277|            12.6|          82728.79|
+----------------+--------------+----------------+------------------+



Export the KPI table

In [68]:
kpi_summary.toPandas().to_csv(
    "../output/kpi_summary.csv",
    index=False
)

print("KPI table exported!")

C:\Users\chatu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\chatu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


KPI table exported!
